# Gaussian mixture models (GMM)

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
We assume the data comes from **several normal (Gaussian) distributions**; each cluster is a Gaussian with mean $\mu_k$, covariance matrix $\Sigma_k$ (the shape and direction of an ellipse) and weight $\pi_k$:

$$p(\mathbf x)=\sum_{k=1}^{K}\pi_k\,\mathcal N(\mathbf x\mid\mu_k,\Sigma_k)$$

Unlike K-Means, which assigns each point **definitely** to one cluster, GMM gives a **membership probability** (soft clustering), and clusters can be **elliptical and stretched**.

> [K-Means: round clusters and hard labels. GMM: stretched ellipses and mixed colors (soft membership).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/gmm.png)

*K-Means: round clusters and hard labels. GMM: stretched ellipses and mixed colors (soft membership).*

## The EM algorithm
Two steps repeat:
- **E (expectation):** the membership probability of each point in each component: $r_{ik}=\frac{\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_j\pi_j\mathcal N(x_i\mid\mu_j,\Sigma_j)}$
- **M (maximization):** recompute the parameters with these weights: $\mu_k=\frac{\sum_ir_{ik}x_i}{\sum_ir_{ik}}$, $\pi_k=\frac1n\sum_ir_{ik}$ and the weighted covariance.

Each EM round never lowers the **likelihood**, so it converges (perhaps to a local optimum).

[[lab:gmm]]

In the lab, press "One EM step" a few times and watch the ellipses settle while the log-likelihood rises.

## A worked E step (1-D)
Two components with $\mu_1=0$, $\mu_2=4$, $\sigma=1$ and $\pi=0.5$. For $x=1$:

| | Density $\mathcal N(1\mid\mu,1)$ | Membership |
|---|---|---|
| Component 1 | 0.242 | 0.242/(0.242+0.0044) = **0.982** |
| Component 2 | 0.0044 | **0.018** |



In [ ]:
import numpy as np
from scipy.stats import norm

x = 1.0
d1, d2 = norm.pdf(x, 0, 1), norm.pdf(x, 4, 1)
r1 = 0.5 * d1 / (0.5 * d1 + 0.5 * d2)
print(f"densities: {d1:.4f}, {d2:.4f}")
print(f"membership: component 1 = {r1:.3f}, component 2 = {1 - r1:.3f}")



## Choosing the number of components: BIC
$\text{BIC}=-2\ln L+p\ln n$, where $p$ is the number of parameters. Choose the model with the **lowest BIC**; the second term penalizes complex models.

| `covariance_type` | Cluster shape |
|---|---|
| `full` | Any ellipse in any direction (default) |
| `tied` | All the same shape |
| `diag` | Axis-parallel ellipses |
| `spherical` | Circles (like K-Means) |



In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

rng = np.random.default_rng(0)
X = np.r_[rng.multivariate_normal([0, 0], [[3, 2.2], [2.2, 2]], 200),
          rng.multivariate_normal([5, 0], [[0.5, 0], [0, 2]], 200),
          rng.multivariate_normal([2, 5], [[1, 0], [0, 0.4]], 200)]
for k in range(1, 7):
    g = GaussianMixture(k, random_state=0).fit(X)
    print(f"K={k}  BIC={g.bic(X):8.1f}")
best = GaussianMixture(3, random_state=0).fit(X)
print("weights:", best.weights_.round(3))
print("soft labels of 3 points:\n", best.predict_proba(X[[0, 250, 500]]).round(3))



> GMM is a **generative model**: `sample()` makes new data that looks like the original, and `score_samples` finds low-probability (anomalous) points.